# Lab 1 — Getting to know your VM, and measuring the Internet

Today's exercise has two goals:

1. Confirm that every tool you will need this semester actually runs in your VM. If something
   is broken, we find out now and not in Lab 3.
2. Run your first real measurements. You will ping and traceroute hosts around the world and
   try to answer a simple question: **can you predict network latency from geographical
   distance?**

The code for measurement and plotting is given to you. Your work is the analysis: reading the
numbers, fitting the model, and explaining where it fails and why.

---

### Rules of the road

You are about to send packets to machines that belong to other people.

- 20 probes per target is polite. A flood is not.
- Unsolicited probing looks like reconnaissance to whoever is on the other end. On a network
  you do not own, that can get you in real trouble.
- The targets in this lab were chosen because they exist to be measured. Do not extend the
  list to random hosts without thinking about it first.
- We will talk more about measurements and active probing during the course, but do not hesitate to ask questions.

## Part 0 — Does the VM work?

Run the cell below. It checks that each tool exists and reports its version. Everything should
come back `OK`. If anything comes back `MISSING`, raise your hand now.

In [ ]:
import shutil, subprocess, sys

TOOLS = [
    ("ping",       ["ping", "-V"]),
    ("traceroute", ["traceroute", "--version"]),
    ("mtr",        ["mtr", "--version"]),
    ("dig",        ["dig", "-v"]),
    ("tshark",     ["tshark", "--version"]),
    ("ip",         ["ip", "-V"]),
    ("tcpdump",    ["tcpdump", "--version"]),
]

for name, cmd in TOOLS:
    if shutil.which(name) is None:
        print(f"{name:12s} MISSING")
        continue
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        first = (out.stdout + out.stderr).strip().split("\n")[0][:70]
        print(f"{name:12s} OK    {first}")
    except Exception as e:
        print(f"{name:12s} ERROR {e}")

print()
print(f"python       OK    {sys.version.split()[0]}")
for mod in ["matplotlib", "numpy", "pandas", "scapy"]:
    try:
        __import__(mod)
        print(f"{mod:12s} OK")
    except ImportError:
        print(f"{mod:12s} MISSING")

### Mininet smoke test

Mininet is the network emulator you will use later in the course. We are not going to learn it
today. We only want proof that it starts, builds a topology and forwards packets.

Run the following command **in a terminal**:

```bash
sudo mn --test pingall
```



You should see `*** Results: 0% dropped (2/2 received)` and then a clean exit. If you see
errors about Open vSwitch, try `sudo service openvswitch-switch start` and run it again.

If mininet leaves things in a bad state at any point this semester, `sudo mn -c` cleans up.

## Part 1 — Your own baseline

Before measuring the world, measure your immediate surroundings. Everything you observe later
includes this cost, so it is worth knowing how big it is.

Find your default gateway (the first router your packets meet) and ping it, plus one host
inside the university network.

In [ ]:
import subprocess, re

def default_gateway():
    out = subprocess.run(["ip", "route", "show", "default"],
                         capture_output=True, text=True).stdout
    m = re.search(r"default via (\S+)", out)
    return m.group(1) if m else None

GATEWAY = default_gateway()
print("Default gateway:", GATEWAY)

print(subprocess.run(["ip", "-brief", "addr"], capture_output=True, text=True).stdout)

Now to the measurements. You will use `ping_host()` for the rest of the lab. Read the script below to verify that you understand its content. We set an interval of 0.2 s between probes so the whole campaign does not take all afternoon :) But do not lower it.

In [ ]:
import subprocess, re
from statistics import mean, stdev

def ping_host(host, count=20, interval=0.2, timeout=2, deadline=30):
    """Send `count` ICMP echo requests. Returns a dict of results.

    rtts   : list of individual RTTs in ms (only the ones that came back)
    loss   : fraction of probes lost, 0.0 to 1.0
    min/avg/max/jitter : summary statistics in ms, or None if nothing came back
    """
    cmd = ["ping", "-c", str(count), "-i", str(interval),
           "-W", str(timeout), "-w", str(deadline), host]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True,
                             timeout=deadline + 10).stdout
    except subprocess.TimeoutExpired:
        out = ""

    rtts = [float(x) for x in re.findall(r"time[=<]([\d.]+)\s*ms", out)]

    m = re.search(r"(\d+) packets transmitted, (\d+) received", out)
    sent, recv = (int(m.group(1)), int(m.group(2))) if m else (count, len(rtts))

    res = {"host": host, "sent": sent, "received": recv, "rtts": rtts,
           "loss": 1 - recv / sent if sent else 1.0, "raw": out}

    if rtts:
        res.update(min=min(rtts), avg=mean(rtts), max=max(rtts),
                   jitter=stdev(rtts) if len(rtts) > 1 else 0.0)
    else:
        res.update(min=None, avg=None, max=None, jitter=None)
    return res


# Try it on the gateway
if GATEWAY:
    r = ping_host(GATEWAY, count=20)
    print(f"{r['host']}  min={r['min']:.3f} ms  avg={r['avg']:.3f} ms  "
          f"max={r['max']:.3f} ms  loss={r['loss']:.0%}")

**Q1.** We ask for **20 probes**, and we work with the **minimum** RTT, not the average. Why do we not use the average in your opinion?

Write your answer here:

**Q2.** What is the minimum RTT to your gateway? Compare it to the average and the maximum.
The gateway is metres away, so propagation delay is essentially zero. What are you actually
measuring? Where does the spread between min and max come from?

Write your answer here:

>

## Part 2 — The physics we are testing against

Light in a vacuum travels at 300,000 km/s. In optical fibre it is slower, roughly two thirds of
that, so about **200,000 km/s**.

A round trip covers the distance twice. So for a distance $d$ in km, the RTT cannot possibly
be below

$$\text{RTT}_{\min} = \frac{2d}{200{,}000 \text{ km/s}} = \frac{d}{100} \text{ ms}$$

**1000 km costs at least 10 ms, round trip.** That is a hard floor set by physics. No amount of
engineering gets you under it.

Everything you measure today will be above this line. The interesting question is: *how far
above, and why?*

In [ ]:
FIBRE_SPEED_KM_S = 200_000     # ~2/3 c, propagation speed in optical fibre

def fibre_bound_ms(distance_km):
    """Theoretical minimum RTT in ms for a straight fibre of the given length."""
    return 2 * distance_km / FIBRE_SPEED_KM_S * 1000

for d in [100, 500, 1000, 5000, 10000, 20000]:
    print(f"{d:6d} km  ->  {fibre_bound_ms(d):7.2f} ms minimum RTT")

## Part 3 — The targets

Below is your target list. These are **RIPE Atlas anchors**: machines maintained by network
operators specifically so that people can measure against them. They sit at known coordinates,
they answer ICMP, and they are not hidden behind a CDN.

That last point matters more than you might think. If you ping `google.com` from Lyon you are
not measuring the distance to California. You are measuring the distance to a cache a few tens
of kilometres away. Anycast and CDNs make a naive distance-vs-latency study meaningless (we'll learn about how this work in class), which
is exactly why the targets here were picked by hand.

In [ ]:
# label, hostname, latitude, longitude
TARGETS = [
    ("Grenoble, FR",     "fr-gnb-as1942-client.anchors.atlas.ripe.net",   45.19,    5.72),
    ("Paris, FR",        "fr-par-as2486.anchors.atlas.ripe.net",          48.86,    2.35),
    ("Amsterdam, NL",    "nl-ams-as1101.anchors.atlas.ripe.net",          52.37,    4.90),
    ("London, UK",       "uk-lon-as5459.anchors.atlas.ripe.net",          51.51,   -0.13),
    ("Stockholm, SE",    "se-sto-as199150.anchors.atlas.ripe.net",        59.33,   18.07),
    ("Athens, GR",       "gr-ath-as199399.anchors.atlas.ripe.net",        37.98,   23.73),
    ("Kampala, UG",      "ug-kla-as37386.anchors.atlas.ripe.net",          0.31,   32.58),
    ("New York, US",     "us-nyc-as14061.anchors.atlas.ripe.net",         40.71,  -74.01),
    ("Sao Paulo, BR",    "br-sao-as401612.anchors.atlas.ripe.net",       -23.55,  -46.63),
    ("Johannesburg, ZA", "za-jnb-as37474.anchors.atlas.ripe.net",        -26.20,   28.05),
    ("Tokyo, JP",        "jp-tyo-as2497.anchors.atlas.ripe.net",          35.68,  139.69),
    ("Singapore, SG",    "sg-sin-as132337.anchors.atlas.ripe.net",         1.35,  103.82),
    ("Sydney, AU",       "au-syd-as4826.anchors.atlas.ripe.net",         -33.87,  151.21),
]

# Where you are measuring from.
ORIGIN = ("Lyon, FR", 45.75, 4.85)

print(f"{len(TARGETS)} targets, measuring from {ORIGIN[0]}")

Anchors do occasionally go offline. Run the check below before starting the campaign: it sends
two probes to each target and tells you which ones are alive. Drop the dead ones from `TARGETS`
before continuing, but keep at least eight, spread across the distance range.

In [ ]:
import socket

alive = []
for label, host, lat, lon in TARGETS:
    try:
        ip = socket.gethostbyname(host)
    except socket.gaierror:
        print(f"  DNS FAIL   {label:18s} {host}")
        continue
    r = ping_host(host, count=2, interval=0.3, deadline=6)
    if r["received"] > 0:
        print(f"  alive      {label:18s} {ip:16s} {r['min']:.1f} ms")
        alive.append((label, host, lat, lon))
    else:
        print(f"  NO REPLY   {label:18s} {ip}")

print(f"\n{len(alive)}/{len(TARGETS)} targets responding")
# TARGETS = alive     # uncomment to drop the dead ones

## Part 4 — Distance

You need the great-circle distance from Lyon to each target: the shortest path over the surface
of the Earth. The haversine formula gives it to you.

This is geometry, not networking, so it is written for you.

In [ ]:
from math import radians, sin, cos, asin, sqrt

EARTH_RADIUS_KM = 6371.0

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points given in decimal degrees."""
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * asin(sqrt(a))


_, olat, olon = ORIGIN
for label, host, lat, lon in TARGETS:
    print(f"{label:18s} {haversine(olat, olon, lat, lon):8.0f} km")

## Part 5 — The ping campaign

This takes a few minutes. It runs the targets one at a time on purpose: if you ping thirteen
hosts in parallel, your own uplink becomes the bottleneck and you end up measuring your VM
rather than the Internet.

In [ ]:
import pandas as pd
import time

rows = []
for label, host, lat, lon in TARGETS:
    print(f"pinging {label:18s} ... ", end="", flush=True)
    r = ping_host(host, count=20, interval=0.2, deadline=30)
    dist = haversine(olat, olon, lat, lon)
    rows.append({
        "label": label, "host": host, "lat": lat, "lon": lon,
        "distance_km": dist,
        "rtt_min": r["min"], "rtt_avg": r["avg"], "rtt_max": r["max"],
        "jitter": r["jitter"], "loss": r["loss"],
        "bound_ms": fibre_bound_ms(dist),
    })
    print("ok" if r["received"] else "no reply")

df = pd.DataFrame(rows)
df["ratio"] = df["rtt_min"] / df["bound_ms"]      # how many times the physical floor
df.to_csv("ping_results.csv", index=False)

df[["label", "distance_km", "rtt_min", "rtt_avg", "jitter",
    "loss", "bound_ms", "ratio"]].round(2)

### Look at the table before you plot anything

**Q2.** The `ratio` column is the measured minimum RTT divided by the theoretical floor. It is
never 1. What is the range you observe? Is it roughly constant across targets, or does it vary?
Which target has the worst ratio, and which has the best?

>

**Q3.** Look at the `jitter` and `loss` columns. Do distant targets have more jitter than close
ones? Should they? Jitter and propagation delay have different causes: say what each one is.

>

## Part 6 — Fit the model

Here is a starting plot: measured minimum RTT against distance, with the fibre bound drawn
underneath.

Your job is to fit a line to the measurements and interpret the two numbers that come out of it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

d = df["distance_km"].values
y = df["rtt_min"].values

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(d, y, s=60, zorder=3, label="measured min RTT")

# the physical floor
grid = np.linspace(0, d.max() * 1.05, 100)
ax.plot(grid, fibre_bound_ms(grid), "k--", lw=1.5,
        label="fibre bound (d / 100)")

for _, row in df.iterrows():
    ax.annotate(row["label"], (row["distance_km"], row["rtt_min"]),
                fontsize=8, xytext=(5, 4), textcoords="offset points")

ax.set_xlabel("great-circle distance from Lyon (km)")
ax.set_ylabel("minimum RTT (ms)")
ax.set_title("Latency vs distance")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TODO: fit a straight line  rtt = slope * distance + intercept
#
# np.polyfit(x, y, 1) returns [slope, intercept].
# Add the fitted line to the plot above, then answer the questions below.

slope, intercept = np.polyfit(d, y, 1)
print(f"rtt_min  =  {slope:.5f} * distance_km  +  {intercept:.2f}")
print()
print(f"slope in ms per 1000 km : {slope * 1000:.2f}")
print(f"fibre bound equivalent  : {fibre_bound_ms(1000):.2f}")
print(f"ratio                   : {slope * 1000 / fibre_bound_ms(1000):.2f}x")

# TODO: how good is the fit? Compute R^2.
resid = y - (slope * d + intercept)
r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"R^2                     : {r2:.3f}")

### Interpreting your fit

**Q4.** Your slope is steeper than the fibre bound. Give **three distinct reasons** why. Think
about the difference between the path your packets take and the straight line you measured on
the globe, and about what happens at each router along the way.

>

**Q5.** Your intercept is not zero. It is the RTT the model predicts for a target at zero
distance. What is physically happening in those milliseconds? Compare the value to what you
measured to your gateway in Part 1.

>

**Q6.** Your $R^2$ is probably high, which makes the model look good. Find the targets with the
largest residuals and name them. For each one, propose an explanation. At least one of them
should have nothing to do with distance.

>

In [ ]:
# Residuals: where does the model fail?
df["predicted"] = slope * df["distance_km"] + intercept
df["residual"] = df["rtt_min"] - df["predicted"]

df[["label", "distance_km", "rtt_min", "predicted", "residual"]] \
    .sort_values("residual", ascending=False).round(1)

**Q7.** Compare two targets at similar distance from Lyon but with clearly different RTT.
(Athens and Tunis are a good pair to start with, but use whatever your data gives you.) Same
kilometres, different milliseconds. What does that tell you about how Internet paths are
actually chosen? Is the shortest path the geographical one?

>

## Part 7 — Traceroute

`ping` tells you the cost of a path. `traceroute` tells you what the path is made of.

The trick is simple and worth understanding. Every IP packet carries a TTL field that each
router decrements. When it hits zero, the router discards the packet and sends back an ICMP
Time Exceeded message. So if you send a packet with TTL=1, the first router complains and
identifies itself. TTL=2 gets you the second. And so on, until you reach the destination.

We run it with `-n` so it does not resolve names, which makes the output much easier to parse
and much faster. We will do the reverse lookups ourselves afterwards, because those names turn
out to be interesting.

In [ ]:
import subprocess, re

IP_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def traceroute(host, max_hops=30, queries=3, wait=2):
    """Run traceroute -n and parse it.

    Returns a list of hops. Each hop is a dict:
      ttl   : the TTL value used
      ips   : set of IPs that answered at this TTL (more than one means load balancing)
      rtts  : list of RTTs in ms, None for a timeout
    """
    cmd = ["traceroute", "-n", "-m", str(max_hops),
           "-q", str(queries), "-w", str(wait), host]
    out = subprocess.run(cmd, capture_output=True, text=True,
                         timeout=max_hops * wait * queries + 30).stdout

    hops = []
    for line in out.splitlines():
        toks = line.split()
        if not toks or not toks[0].isdigit():
            continue
        hop = {"ttl": int(toks[0]), "ips": [], "rtts": []}
        current_ip = None
        i = 1
        while i < len(toks):
            t = toks[i]
            if t == "*":
                hop["rtts"].append(None)
            elif IP_RE.match(t):
                current_ip = t
                if t not in hop["ips"]:
                    hop["ips"].append(t)
            elif t == "ms":
                pass
            else:
                try:
                    hop["rtts"].append(float(t))
                except ValueError:
                    pass          # MPLS labels, !H, !N and friends
            i += 1
        hops.append(hop)
    return hops


def hop_summary(hops):
    """Best RTT per hop, and the number of hops that answered at all."""
    out = []
    for h in hops:
        vals = [r for r in h["rtts"] if r is not None]
        out.append({"ttl": h["ttl"],
                    "ip": h["ips"][0] if h["ips"] else None,
                    "all_ips": h["ips"],
                    "rtt": min(vals) if vals else None})
    return out

### Before you look at the output, know what is normal

Traceroute output is full of things that look like failures and are not:

- **Stars (`* * *`)** usually mean the router is configured not to answer, or rate-limits ICMP.
  Your packet got through fine. It just did not get a reply from that particular hop. A trace
  full of stars in the middle that still reaches the destination is completely normal.
- **The same hop appearing with two different IPs** means load balancing. Consecutive probes
  took different physical paths.
- **RTT going *down* from one hop to the next** happens. Each measurement is an independent
  round trip, and the return path may differ from hop to hop.
- **Per-hop RTT is a round trip**, forward path plus return path. The return path is often not
  the reverse of the forward path. This is why you cannot treat a traceroute as a map.
- **MPLS tunnels** hide entire sections of the path. Several routers can be crossed while the
  TTL stays untouched.

Now run it.

In [ ]:
traces = {}
for label, host, lat, lon in TARGETS:
    print(f"tracing {label:18s} ... ", end="", flush=True)
    try:
        traces[label] = traceroute(host)
        print(f"{len(traces[label])} hops")
    except Exception as e:
        traces[label] = []
        print("failed:", e)

In [ ]:
# Look at one trace in full
LOOK_AT = "Tokyo, JP"     # change this

for h in hop_summary(traces[LOOK_AT]):
    rtt = f"{h['rtt']:8.2f} ms" if h["rtt"] is not None else "       *   "
    extra = f"   (+{len(h['all_ips'])-1} more IPs)" if len(h["all_ips"]) > 1 else ""
    print(f"{h['ttl']:3d}  {str(h['ip'] or '*'):16s} {rtt}{extra}")

### Hop count against distance

Now the plot you might expect to work, and probably will not.

In [ ]:
df["hops"] = [len([h for h in traces[l] if h["ips"]]) if traces[l] else np.nan
              for l in df["label"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df["distance_km"], df["hops"], s=60)
axes[0].set_xlabel("distance (km)")
axes[0].set_ylabel("responding hops")
axes[0].set_title("Hop count vs distance")
axes[0].grid(alpha=0.3)

axes[1].scatter(df["hops"], df["rtt_min"], s=60)
axes[1].set_xlabel("responding hops")
axes[1].set_ylabel("min RTT (ms)")
axes[1].set_title("RTT vs hop count")
axes[1].grid(alpha=0.3)

for _, row in df.iterrows():
    if np.isnan(row["hops"]):
        continue
    short = row["label"].split(",")[0]
    axes[0].annotate(short, (row["distance_km"], row["hops"]),
                     fontsize=8, xytext=(5, 4), textcoords="offset points")
    axes[1].annotate(short, (row["hops"], row["rtt_min"]),
                     fontsize=8, xytext=(5, 4), textcoords="offset points")

plt.tight_layout()
plt.show()

# TODO: compute the correlation of hop count with distance, and compare it
# to the correlation of RTT with distance.
print("corr(hops, distance) :", df[["hops", "distance_km"]].corr().iloc[0, 1].round(3))
print("corr(rtt,  distance) :", df[["rtt_min", "distance_km"]].corr().iloc[0, 1].round(3))

**Q8.** Distance predicts RTT reasonably well. Does it predict hop count? Compare the two
correlations you just computed.

>

**Q9.** A hop is a router. Routers are placed where they are for reasons that have very little
to do with geography. Given that, what do you think hop count actually measures? Find a case in
your data where two targets at very different distances have almost the same hop count, and one
where two targets at similar distance have very different hop counts.

>

## Part 8 — Finding the ocean

Within a trace, look at how the RTT grows hop by hop. Most hops add a fraction of a millisecond.
Then somewhere there is one hop that adds tens of milliseconds all at once. That is a long-haul
link: a submarine cable, or a long terrestrial backbone span.

You can find it without knowing anything about the network. Just look for the jump.

In [ ]:
def hop_deltas(hops):
    """RTT increase from one responding hop to the next."""
    seq = [h for h in hop_summary(hops) if h["rtt"] is not None]
    out = []
    for prev, cur in zip(seq, seq[1:]):
        out.append({"from_ttl": prev["ttl"], "to_ttl": cur["ttl"],
                    "from_ip": prev["ip"], "to_ip": cur["ip"],
                    "delta": cur["rtt"] - prev["rtt"]})
    return out


for label in df["label"]:
    if not traces.get(label):
        continue
    ds = hop_deltas(traces[label])
    if not ds:
        continue
    big = max(ds, key=lambda x: x["delta"])
    print(f"{label:18s} biggest jump +{big['delta']:6.1f} ms  "
          f"at hop {big['from_ttl']}->{big['to_ttl']}  "
          f"{big['from_ip']} -> {big['to_ip']}")

### Reading router names

Network operators name their routers systematically, and the names usually contain an airport
or city code. `par` is Paris, `fra` Frankfurt, `ams` Amsterdam, `lon` London, `nyc` New York,
`mrs` Marseille, `sin` Singapore, `nrt` Tokyo Narita.

This means you can often reconstruct the geography of a path from DNS alone.

In [ ]:
import socket

def reverse_dns(ip):
    try:
        return socket.gethostbyaddr(ip)[0]
    except Exception:
        return None

# Names around the biggest jump, for one target
LOOK_AT = "Tokyo, JP"     # change this

seq = [h for h in hop_summary(traces[LOOK_AT]) if h["rtt"] is not None]
for h in seq:
    name = reverse_dns(h["ip"]) or ""
    print(f"{h['ttl']:3d}  {h['ip']:16s} {h['rtt']:8.2f} ms   {name}")

**Q10.** For the three most distant targets: where is the biggest RTT jump, and what do the
router names on either side of it tell you? Sketch the route on a map (paper is fine). Does the
path go where you expected?

>

**Q11.** Take your traceroute to Tunis or Johannesburg. Does the path go straight south, or does
it go north first? If it goes north, explain why that might be the case. This is not a technical
question.

>

**Q12.** Pick one long trace and estimate the length of the longest link from its RTT delta,
using the fibre bound in reverse. Compare your estimate to a submarine cable map
(<https://www.submarinecablemap.com>). How close are you?

>